# Clothing Classification Model Evaluation (Thesis Edition)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_REPO/evaluate_model_thesis_colab.ipynb)

**Generate publication-quality evaluation figures WITHOUT retraining!**

This notebook:
- ✅ Loads `clothing_model_final.h5` from Google Drive automatically
- ✅ Generates predictions on validation dataset
- ✅ Creates comprehensive thesis evaluation figures (5 publication-quality charts)
- ✅ Exports metrics in JSON, CSV, and LaTeX formats
- ✅ All figures at 300 DPI (publication quality)
- ✅ **All results saved to Google Drive** (organized by timestamp)
- ✅ Optional: Download all results as a zip file

**No training required - just evaluation!**

## What Gets Saved to Drive:

```
📁 Google Drive/clothing_classification_thesis/
  └── evaluation_results/
      └── YYYYMMDD_HHMMSS/          # Timestamped folder
          ├── figures/               # All 5 figures (PNG + PDF)
          ├── metrics/               # JSON, CSV, LaTeX files
          ├── predictions/           # Raw predictions data
          └── EVALUATION_SUMMARY.txt # Complete summary report
```

## 1. Setup Google Colab Environment

In [ ]:
# Check GPU availability (optional for evaluation, but faster)
import tensorflow as tf

print("="*60)
print("Google Colab Environment Check")
print("="*60)
print(f"TensorFlow version: {tf.__version__}")
print(f"Python version: {__import__('sys').version}")

# Check for GPU
gpu_devices = tf.config.list_physical_devices('GPU')
if gpu_devices:
    print(f"\n✅ GPU available: {len(gpu_devices)} device(s)")
    print("🚀 Evaluation will use GPU acceleration (faster)")
else:
    print("\n⚠️ No GPU found. Evaluation will use CPU (still works fine)")

print("="*60)

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
import os
from datetime import datetime

# Mount Google Drive
drive.mount('/content/drive')

# Project directories
DRIVE_ROOT = '/content/drive/MyDrive/clothing_classification_thesis'
DATASET_DIR = f'{DRIVE_ROOT}/dataset'
MODELS_DIR = f'{DRIVE_ROOT}/trained_models'

# Create timestamped output directory for this evaluation run
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_DIR = f'{DRIVE_ROOT}/evaluation_results/{timestamp}'
FIGURES_DIR = f'{OUTPUT_DIR}/figures'
METRICS_DIR = f'{OUTPUT_DIR}/metrics'
PREDICTIONS_DIR = f'{OUTPUT_DIR}/predictions'

# Create all output directories
for dir_path in [DRIVE_ROOT, DATASET_DIR, MODELS_DIR, OUTPUT_DIR, 
                 FIGURES_DIR, METRICS_DIR, PREDICTIONS_DIR]:
    os.makedirs(dir_path, exist_ok=True)

print("="*70)
print("✅ Google Drive Mounted Successfully")
print("="*70)
print(f"\n📁 Directory Structure:")
print(f"   Root:        {DRIVE_ROOT}")
print(f"   Dataset:     {DATASET_DIR}")
print(f"   Models:      {MODELS_DIR}")
print(f"   Output:      {OUTPUT_DIR}")
print(f"   ├── Figures:     {FIGURES_DIR}")
print(f"   ├── Metrics:     {METRICS_DIR}")
print(f"   └── Predictions: {PREDICTIONS_DIR}")
print("\n💡 All results will be saved to your Google Drive")
print("="*70)

## 3. Install Required Packages

In [ ]:
# Install packages
!pip install -q scikit-learn matplotlib seaborn scipy pandas

# Import libraries
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img
from tensorflow.keras.models import load_model
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    log_loss, roc_auc_score, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    brier_score_loss, cohen_kappa_score
)

# Set publication-quality plot style
plt.style.use('seaborn-v0_8-paper')
sns.set_palette("husl")
plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.titlesize': 16,
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
})

print("✅ All packages imported successfully")

## 4. Download Pre-trained Model (clothing_model_final.h5)

**Automatically downloads the latest trained model from Google Drive**

Model file: `clothing_model_final.h5` - No training required!

In [ ]:
# Install gdown for downloading from Google Drive
!pip install -q gdown

import gdown

# Configuration - Download clothing_model_final.h5
MODEL_FILE_URL = 'https://drive.google.com/uc?id=1fWY0GbU-5JwUVtFky_zQ4HFFI5ibO2O8'
MODEL_FILE_NAME = 'clothing_model_final.h5'
MODEL_PATH = f'{MODELS_DIR}/{MODEL_FILE_NAME}'

print("="*70)
print("Downloading Pre-trained Model")
print("="*70)

# Download model file if not already present
if not os.path.exists(MODEL_PATH):
    print(f"📥 Downloading {MODEL_FILE_NAME} from Google Drive...")
    print("   This may take a few minutes...\n")
    gdown.download(MODEL_FILE_URL, MODEL_PATH, quiet=False)
    print(f"\n✅ Downloaded: {MODEL_PATH}")
else:
    print(f"✅ Model file already exists: {MODEL_PATH}")

# Check model file size
if os.path.exists(MODEL_PATH):
    model_size_mb = os.path.getsize(MODEL_PATH) / (1024 * 1024)
    print(f"   Size: {model_size_mb:.2f} MB")
    print(f"\n✅ Model ready: {MODEL_FILE_NAME}")
else:
    print(f"\n❌ Model file not found: {MODEL_PATH}")
    print(f"   Please check your Google Drive connection and file ID")

# Also download additional config files if available (from old zip)
CONFIG_FILES_URL = 'https://drive.google.com/uc?id=1oZhdDnXcQs5Oy84z1AqxoGV9GxEVKUbH'
CONFIG_ZIP_PATH = '/content/trained_models.zip'

print(f"\n📦 Checking for additional config files...")
if not os.path.exists(f'{MODELS_DIR}/class_labels.json'):
    print("📥 Downloading config files (class_labels.json, etc.)...")
    try:
        gdown.download(CONFIG_FILES_URL, CONFIG_ZIP_PATH, quiet=False)
        
        import zipfile
        with zipfile.ZipFile(CONFIG_ZIP_PATH, 'r') as zip_ref:
            # Extract only JSON files
            for fname in zip_ref.namelist():
                if fname.endswith('.json'):
                    zip_ref.extract(fname, MODELS_DIR)
                    print(f"   ✅ Extracted: {fname}")
        
        os.remove(CONFIG_ZIP_PATH)
        print(f"✅ Config files extracted")
    except Exception as e:
        print(f"⚠️  Could not download config files: {e}")
        print(f"   Will use default configurations")
else:
    print(f"✅ Config files already exist")

print("="*70)
print(f"✅ Model setup complete!")
print(f"   Model file: {MODEL_PATH}")
print("="*70)

## 5. Download Dataset (if needed)

The validation dataset is required for generating predictions and evaluation figures.

In [ ]:
# Dataset configuration
DATASET_URL = 'https://drive.google.com/uc?id=1mqnxghcVu2trYKdbQRnd93OgvrEQJeZW'
DATASET_ZIP_PATH = f'{DATASET_DIR}/clothing.zip'
DATASET_ROOT = f'{DATASET_DIR}/clothing_dataset'

print("="*60)
print("Checking Dataset")
print("="*60)

# Check if dataset already exists
if os.path.exists(DATASET_ROOT):
    print(f"✅ Dataset already exists: {DATASET_ROOT}")
    
    # Count images
    subdirs = [d for d in os.listdir(DATASET_ROOT) if os.path.isdir(os.path.join(DATASET_ROOT, d))]
    print(f"\n📊 Dataset classes: {subdirs}")
    
    total_images = 0
    for cls in subdirs:
        cls_path = os.path.join(DATASET_ROOT, cls)
        n_images = len([f for f in os.listdir(cls_path) 
                       if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff'))])
        total_images += n_images
        print(f"   {cls:12s}: {n_images:5d} images")
    print(f"\n   Total images: {total_images}")
    
else:
    print(f"⚠️  Dataset not found. Downloading...")
    print(f"📥 Downloading clothing dataset from Google Drive...")
    print(f"   This may take a few minutes...\n")
    
    # Create dataset directory
    os.makedirs(DATASET_DIR, exist_ok=True)
    
    # Download dataset
    gdown.download(DATASET_URL, DATASET_ZIP_PATH, quiet=False)
    print(f"\n✅ Dataset downloaded: {DATASET_ZIP_PATH}")
    
    # Extract dataset
    print(f"\n📦 Extracting dataset...")
    with zipfile.ZipFile(DATASET_ZIP_PATH, 'r') as z:
        z.extractall(DATASET_ROOT)
    print(f"✅ Dataset extracted to: {DATASET_ROOT}")
    
    # Clean up zip
    os.remove(DATASET_ZIP_PATH)
    print(f"✅ Removed: {DATASET_ZIP_PATH}")

print("="*60)
print("✅ Dataset ready!")
print("="*60)

## 6. Load Model and Configuration

In [ ]:
# Load class labels
try:
    with open(f'{MODELS_DIR}/class_labels.json', 'r') as f:
        class_indices = json.load(f)
    # Sort by index to get correct order
    classes = [k for k, _ in sorted(class_indices.items(), key=lambda x: x[1])]
    print(f"✅ Loaded class labels: {classes}")
except:
    # Fallback to default order (MUST match your training!)
    classes = ['trousers', 'tshirt', 'other']
    print(f"⚠️  Using default class order: {classes}")

num_classes = len(classes)

# Load model
print(f"\n📦 Loading model from: {MODEL_PATH}")
model = load_model(MODEL_PATH, compile=False)

print("\n✅ Model loaded successfully!")
print(f"   Input shape:  {model.input_shape}")
print(f"   Output shape: {model.output_shape}")
print(f"   Classes: {num_classes}")

## 7. Create Validation Data Generator

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Create validation generator (no augmentation)
val_datagen = ImageDataGenerator(rescale=1./255)

val_gen = val_datagen.flow_from_directory(
    DATASET_ROOT,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False,  # Important for evaluation!
    seed=42,
    classes=classes
)

print("="*60)
print("Validation Data Generator")
print("="*60)
print(f"Total samples: {val_gen.samples}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Image size: {IMG_SIZE}")
print(f"Class indices: {val_gen.class_indices}")
print("="*60)

## 8. Generate Predictions on Validation Set

In [ ]:
print("🔮 Generating predictions on validation set...")
print("   This may take a few minutes...\n")

# Generate predictions
y_prob = model.predict(val_gen, verbose=1)
y_pred = np.argmax(y_prob, axis=1)
y_true = val_gen.classes

# Get one-hot encoded labels
y_true_onehot = np.eye(num_classes)[y_true]

# Calculate basic masks
correct_mask = y_pred == y_true
max_probs = y_prob.max(axis=1)

# Compute confusion matrix
cm = confusion_matrix(y_true, y_pred)
cm_normalized = cm.astype('float') / (cm.sum(axis=1, keepdims=True) + 1e-12)

print(f"\n✅ Predictions complete")
print(f"   Total samples: {len(y_true)}")
print(f"   Correct predictions: {correct_mask.sum()} ({100*correct_mask.mean():.2f}%)")
print(f"   Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f"\n📊 Predictions per class:")
for i, cls in enumerate(classes):
    n_pred = (y_pred == i).sum()
    n_true = (y_true == i).sum()
    print(f"   {cls:12s}: {n_pred:4d} predicted, {n_true:4d} actual")

## 9. Figure 1: Confusion Matrices

In [ ]:
print("📈 Generating Figure 1: Confusion Matrices...")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[c.capitalize() for c in classes],
            yticklabels=[c.capitalize() for c in classes],
            ax=axes[0], cbar_kws={'label': 'Count'},
            linewidths=0.5, linecolor='gray')
axes[0].set_title('Confusion Matrix (Counts)', fontweight='bold', pad=15)
axes[0].set_xlabel('Predicted Label', fontweight='bold')
axes[0].set_ylabel('True Label', fontweight='bold')

# Normalized
sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Greens',
            xticklabels=[c.capitalize() for c in classes],
            yticklabels=[c.capitalize() for c in classes],
            ax=axes[1], cbar_kws={'label': 'Percentage'},
            linewidths=0.5, linecolor='gray', vmin=0, vmax=1)
axes[1].set_title('Confusion Matrix (Normalized)', fontweight='bold', pad=15)
axes[1].set_xlabel('Predicted Label', fontweight='bold')
axes[1].set_ylabel('True Label', fontweight='bold')

plt.tight_layout()

# Save to Google Drive
fig1_png = f'{FIGURES_DIR}/fig1_confusion_matrices.png'
fig1_pdf = f'{FIGURES_DIR}/fig1_confusion_matrices.pdf'
plt.savefig(fig1_png, dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(fig1_pdf, bbox_inches='tight')
plt.show()

print(f"✅ Saved to Drive: {fig1_png}")
print(f"✅ Saved to Drive: {fig1_pdf}")

## 10. Figure 2: ROC Curves

In [ ]:
print("📈 Generating Figure 2: ROC Curves...")

fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.Set2(np.linspace(0, 1, num_classes))

for i, (cls, color) in enumerate(zip(classes, colors)):
    fpr, tpr, _ = roc_curve(y_true_onehot[:, i], y_prob[:, i])
    roc_auc_cls = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2.5,
            label=f'{cls.capitalize()} (AUC = {roc_auc_cls:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1.5, alpha=0.5, label='Random Classifier')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', fontweight='bold', fontsize=12)
ax.set_ylabel('True Positive Rate', fontweight='bold', fontsize=12)
ax.set_title('ROC Curves (One-vs-Rest)', fontweight='bold', fontsize=14, pad=15)
ax.legend(loc='lower right', fontsize=11, frameon=True, shadow=True)
ax.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()

# Save to Google Drive
fig2_png = f'{FIGURES_DIR}/fig2_roc_curves.png'
fig2_pdf = f'{FIGURES_DIR}/fig2_roc_curves.pdf'
plt.savefig(fig2_png, dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(fig2_pdf, bbox_inches='tight')
plt.show()

print(f"✅ Saved to Drive: {fig2_png}")
print(f"✅ Saved to Drive: {fig2_pdf}")

## 11. Figure 3: Precision-Recall Curves

In [ ]:
print("📈 Generating Figure 3: Precision-Recall Curves...")

fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.Set2(np.linspace(0, 1, num_classes))

for i, (cls, color) in enumerate(zip(classes, colors)):
    precision, recall, _ = precision_recall_curve(y_true_onehot[:, i], y_prob[:, i])
    ap = average_precision_score(y_true_onehot[:, i], y_prob[:, i])
    ax.plot(recall, precision, color=color, lw=2.5,
            label=f'{cls.capitalize()} (AP = {ap:.3f})')

ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('Recall', fontweight='bold', fontsize=12)
ax.set_ylabel('Precision', fontweight='bold', fontsize=12)
ax.set_title('Precision-Recall Curves', fontweight='bold', fontsize=14, pad=15)
ax.legend(loc='lower left', fontsize=11, frameon=True, shadow=True)
ax.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()

# Save to Google Drive
fig3_png = f'{FIGURES_DIR}/fig3_precision_recall.png'
fig3_pdf = f'{FIGURES_DIR}/fig3_precision_recall.pdf'
plt.savefig(fig3_png, dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(fig3_pdf, bbox_inches='tight')
plt.show()

print(f"✅ Saved to Drive: {fig3_png}")
print(f"✅ Saved to Drive: {fig3_pdf}")

## 12. Figure 4: Per-Class Metrics Bar Chart

In [ ]:
print("📈 Generating Figure 4: Per-Class Metrics...")

# Compute classification report
class_report = classification_report(y_true, y_pred, target_names=classes, 
                                     output_dict=True, zero_division=0)

fig, ax = plt.subplots(figsize=(12, 7))
x = np.arange(len(classes))
width = 0.25

precision_vals = [class_report[cls]['precision'] for cls in classes]
recall_vals = [class_report[cls]['recall'] for cls in classes]
f1_vals = [class_report[cls]['f1-score'] for cls in classes]

bars1 = ax.bar(x - width, precision_vals, width, label='Precision', 
               color='#3498db', edgecolor='black', linewidth=0.7)
bars2 = ax.bar(x, recall_vals, width, label='Recall', 
               color='#2ecc71', edgecolor='black', linewidth=0.7)
bars3 = ax.bar(x + width, f1_vals, width, label='F1-Score', 
               color='#e74c3c', edgecolor='black', linewidth=0.7)

# Add value labels on bars
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2., height, f'{height:.3f}',
                ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xlabel('Class', fontweight='bold', fontsize=12)
ax.set_ylabel('Score', fontweight='bold', fontsize=12)
ax.set_title('Per-Class Performance Metrics', fontweight='bold', fontsize=14, pad=15)
ax.set_xticks(x)
ax.set_xticklabels([cls.capitalize() for cls in classes], fontsize=11)
ax.legend(fontsize=11, frameon=True, shadow=True, loc='lower right')
ax.set_ylim([0, 1.1])
ax.grid(True, axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()

# Save to Google Drive
fig4_png = f'{FIGURES_DIR}/fig4_per_class_metrics.png'
fig4_pdf = f'{FIGURES_DIR}/fig4_per_class_metrics.pdf'
plt.savefig(fig4_png, dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(fig4_pdf, bbox_inches='tight')
plt.show()

print(f"✅ Saved to Drive: {fig4_png}")
print(f"✅ Saved to Drive: {fig4_pdf}")

## 13. Figure 5: Sample Predictions

In [ ]:
print("📈 Generating Figure 5: Sample Predictions...")

# Get correct and incorrect predictions
correct_indices = np.where(correct_mask)[0]
incorrect_indices = np.where(~correct_mask)[0]

n_samples = min(6, len(correct_indices), len(incorrect_indices))

# Randomly sample
if len(correct_indices) >= n_samples:
    correct_samples = np.random.choice(correct_indices, n_samples, replace=False)
else:
    correct_samples = correct_indices

if len(incorrect_indices) >= n_samples:
    incorrect_samples = np.random.choice(incorrect_indices, n_samples, replace=False)
else:
    incorrect_samples = incorrect_indices

# Plot
fig, axes = plt.subplots(2, n_samples, figsize=(18, 7))

# Correct predictions
for idx, sample_idx in enumerate(correct_samples):
    img = load_img(val_gen.filepaths[sample_idx], target_size=IMG_SIZE)
    pred_label = classes[y_pred[sample_idx]]
    confidence = y_prob[sample_idx].max()
    axes[0, idx].imshow(img)
    axes[0, idx].axis('off')
    axes[0, idx].set_title(f'✓ {pred_label.upper()}\n{confidence:.1%}',
                           color='green', fontweight='bold', fontsize=10)

# Incorrect predictions
for idx, sample_idx in enumerate(incorrect_samples):
    img = load_img(val_gen.filepaths[sample_idx], target_size=IMG_SIZE)
    true_label = classes[y_true[sample_idx]]
    pred_label = classes[y_pred[sample_idx]]
    confidence = y_prob[sample_idx].max()
    axes[1, idx].imshow(img)
    axes[1, idx].axis('off')
    axes[1, idx].set_title(f'✗ Pred: {pred_label}\nTrue: {true_label}\n{confidence:.1%}',
                           color='red', fontweight='bold', fontsize=9)

# Add row labels
fig.text(0.02, 0.73, 'Correct\nPredictions', ha='center', va='center',
         fontsize=13, fontweight='bold', color='green', rotation=90)
fig.text(0.02, 0.27, 'Incorrect\nPredictions', ha='center', va='center',
         fontsize=13, fontweight='bold', color='red', rotation=90)

plt.suptitle('Sample Predictions', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout(rect=[0.03, 0, 1, 0.96])

# Save to Google Drive
fig5_png = f'{FIGURES_DIR}/fig5_sample_predictions.png'
plt.savefig(fig5_png, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print(f"✅ Saved to Drive: {fig5_png}")

## 14. Compute and Save All Metrics

In [ ]:
print("📊 Computing comprehensive metrics...")

# Compute all metrics
accuracy = accuracy_score(y_true, y_pred)
logloss = log_loss(y_true, y_prob)
brier = np.mean((y_prob - y_true_onehot)**2)
kappa = cohen_kappa_score(y_true, y_pred)

# Compute per-class AUC scores
per_class_auc = {}
for i, cls in enumerate(classes):
    try:
        roc_auc_cls = roc_auc_score(y_true_onehot[:, i], y_prob[:, i])
        per_class_auc[cls] = float(roc_auc_cls)
    except:
        per_class_auc[cls] = None

# Save comprehensive results
results = {
    'evaluation_timestamp': timestamp,
    'model_file': MODEL_FILE_NAME,
    'total_samples': int(len(y_true)),
    'overall_metrics': {
        'accuracy': float(accuracy),
        'log_loss': float(logloss),
        'brier_score': float(brier),
        'cohen_kappa': float(kappa)
    },
    'per_class_metrics': {cls: {
        'precision': float(class_report[cls]['precision']),
        'recall': float(class_report[cls]['recall']),
        'f1_score': float(class_report[cls]['f1-score']),
        'support': int(class_report[cls]['support']),
        'auc': per_class_auc.get(cls, None)
    } for cls in classes},
    'confusion_matrix': {
        'raw': cm.tolist(),
        'normalized': cm_normalized.tolist()
    }
}

# Save to Google Drive
results_path = f'{METRICS_DIR}/evaluation_results.json'
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f"✅ Saved to Drive: {results_path}")

# Generate LaTeX table
latex_table = r"""\begin{table}[h]
\centering
\caption{Per-Class Performance Metrics}
\label{tab:metrics}
\begin{tabular}{lcccc}
\hline
\textbf{Class} & \textbf{Precision} & \textbf{Recall} & \textbf{F1-Score} & \textbf{Support} \\
\hline
"""

for cls in classes:
    m = class_report[cls]
    latex_table += f"{cls.capitalize()} & {m['precision']:.4f} & {m['recall']:.4f} & {m['f1-score']:.4f} & {int(m['support'])} \\\\\n"

latex_table += r"""\hline
\end{tabular}
\end{table}"""

latex_path = f'{METRICS_DIR}/metrics_table.tex'
with open(latex_path, 'w') as f:
    f.write(latex_table)

print(f"✅ Saved to Drive: {latex_path}")

# Save classification report as CSV
import pandas as pd
report_df = pd.DataFrame(class_report).transpose()
csv_path = f'{METRICS_DIR}/classification_report.csv'
report_df.to_csv(csv_path)
print(f"✅ Saved to Drive: {csv_path}")

# Save predictions for further analysis
predictions_data = {
    'true_labels': y_true.tolist(),
    'predicted_labels': y_pred.tolist(),
    'probabilities': y_prob.tolist(),
    'class_names': classes,
    'filepaths': val_gen.filepaths
}

predictions_path = f'{PREDICTIONS_DIR}/predictions.json'
with open(predictions_path, 'w') as f:
    json.dump(predictions_data, f, indent=2)

print(f"✅ Saved to Drive: {predictions_path}")

## 15. Final Summary

In [ ]:
print("="*70)
print("✨ THESIS EVALUATION COMPLETE!")
print("="*70)

print(f"\n📊 Generated Figures ({FIGURES_DIR}):")
print(f"   1. fig1_confusion_matrices.png / .pdf")
print(f"   2. fig2_roc_curves.png / .pdf")
print(f"   3. fig3_precision_recall.png / .pdf")
print(f"   4. fig4_per_class_metrics.png / .pdf")
print(f"   5. fig5_sample_predictions.png")

print(f"\n📄 Metrics Files ({METRICS_DIR}):")
print(f"   - evaluation_results.json (comprehensive metrics)")
print(f"   - metrics_table.tex (LaTeX table)")
print(f"   - classification_report.csv (CSV format)")

print(f"\n🔮 Predictions Data ({PREDICTIONS_DIR}):")
print(f"   - predictions.json (all predictions + probabilities)")

print(f"\n📁 Main Output Location:")
print(f"   {OUTPUT_DIR}")

print(f"\n📈 Overall Metrics:")
print(f"   Accuracy:      {accuracy:.4f}")
print(f"   Log Loss:      {logloss:.4f}")
print(f"   Brier Score:   {brier:.6f}")
print(f"   Cohen's Kappa: {kappa:.4f}")

print(f"\n📋 Per-Class Performance:")
for cls in classes:
    m = class_report[cls]
    auc_val = per_class_auc.get(cls, 0)
    print(f"   {cls.capitalize():12s}: P={m['precision']:.3f}, R={m['recall']:.3f}, F1={m['f1-score']:.3f}, AUC={auc_val:.3f}")

print("\n💡 All files saved to your Google Drive!")
print("💡 Figures are publication-quality (300 DPI)")
print("💡 Both PNG and PDF formats saved (where applicable)")

print("\n" + "="*70)
print("📥 All results are now in your Google Drive")
print(f"📂 Navigate to: {DRIVE_ROOT}")
print("="*70)

print(f"\n🎉 Evaluation complete! Your thesis materials are ready!")

# Create a summary file
summary_path = f'{OUTPUT_DIR}/EVALUATION_SUMMARY.txt'
with open(summary_path, 'w') as f:
    f.write("="*70 + "\n")
    f.write("CLOTHING CLASSIFICATION MODEL EVALUATION SUMMARY\n")
    f.write("="*70 + "\n\n")
    f.write(f"Evaluation Date: {timestamp}\n")
    f.write(f"Model File: {MODEL_FILE_NAME}\n")
    f.write(f"Total Samples: {len(y_true)}\n\n")
    f.write("OVERALL METRICS:\n")
    f.write(f"  Accuracy:      {accuracy:.6f}\n")
    f.write(f"  Log Loss:      {logloss:.6f}\n")
    f.write(f"  Brier Score:   {brier:.6f}\n")
    f.write(f"  Cohen's Kappa: {kappa:.6f}\n\n")
    f.write("PER-CLASS PERFORMANCE:\n")
    for cls in classes:
        m = class_report[cls]
        auc_val = per_class_auc.get(cls, 0)
        f.write(f"  {cls.capitalize():12s}: Precision={m['precision']:.4f}, Recall={m['recall']:.4f}, F1={m['f1-score']:.4f}, AUC={auc_val:.4f}, Support={int(m['support'])}\n")
    f.write("\n" + "="*70 + "\n")
    f.write("FILES GENERATED:\n")
    f.write("="*70 + "\n")
    f.write("\nFigures:\n")
    f.write("  - fig1_confusion_matrices.png / .pdf\n")
    f.write("  - fig2_roc_curves.png / .pdf\n")
    f.write("  - fig3_precision_recall.png / .pdf\n")
    f.write("  - fig4_per_class_metrics.png / .pdf\n")
    f.write("  - fig5_sample_predictions.png\n")
    f.write("\nMetrics:\n")
    f.write("  - evaluation_results.json\n")
    f.write("  - metrics_table.tex\n")
    f.write("  - classification_report.csv\n")
    f.write("\nPredictions:\n")
    f.write("  - predictions.json\n")

print(f"\n✅ Summary saved to: {summary_path}")

## 16. Optional: Download Results as Zip

In [ ]:
from google.colab import files
import zipfile
import os

print("📦 Creating downloadable zip file from all evaluation results...")

zip_filename = f'/content/thesis_evaluation_{timestamp}.zip'

# Count files
total_files = 0
for root, dirs, file_list in os.walk(OUTPUT_DIR):
    total_files += len(file_list)

print(f"   Found {total_files} files to zip...")

with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, file_list in os.walk(OUTPUT_DIR):
        for fname in file_list:
            fpath = os.path.join(root, fname)
            arcname = os.path.relpath(fpath, OUTPUT_DIR)
            zipf.write(fpath, arcname)
            print(f"  ✅ Added: {arcname}")

zip_size_mb = os.path.getsize(zip_filename) / (1024 * 1024)
print(f"\n✅ Zip file created: {zip_filename} ({zip_size_mb:.2f} MB)")

print("\n💾 Starting download to your local machine...")
print("   Note: All files are also saved in your Google Drive!")
print(f"   Drive location: {OUTPUT_DIR}")

files.download(zip_filename)

print("\n🎉 Download complete! Check your browser's download folder.")
print("\n💡 You can also access all files directly from Google Drive:")
print(f"   {DRIVE_ROOT}")